# R1.4 — статистика и контроллер: малые регрессии
Все входы синтетические. Это не рассчитанная карта и не проверка реальной системы. Пороговые требования не ослабляются.

In [ ]:
from pathlib import Path
import os,sys,json
root=Path.cwd()
while root!=root.parent and not (root/'efc/model.py').exists(): root=root.parent
assert (root/'efc/model.py').exists()
os.chdir(root);sys.path.insert(0,str(root));sys.path.insert(0,str(root/'tests'))
from efc.statistics import *
from efc.pilot import *
from efc.model import Config,simulate
out=root/'results/r14/current/statistics';out.mkdir(parents=True,exist_ok=True)

Ручной ответ: задержки 100/200/300 мс, одна потеря из четырёх поступлений. Объединённый p95 = 290 мс, потери = 0,25, H(200 мс) = 0,5; p95 разных seed не усредняется.

In [ ]:
a=Cohort(0,2,1,0,(100_000_000,));b=Cohort(1,2,0,0,(200_000_000,300_000_000))
p=pooled_metrics([a,b]);assert (p['p95_ms'],p['drop_fraction'],p['H200'])==(290,.25,.5)
from test_statistics import ten
zero=cluster_summary(ten(lost=0));assert classify(zero,warmup_ok=True)=='INSUFFICIENT_DATA'
boundary=cluster_summary(ten(delay=200,arrivals=1000,lost=1));assert classify(boundary,warmup_ok=True)=='ACCEPTABLE'
(out/'statistics-oracles.json').write_text(json.dumps({'scope':'synthetic literal oracles','pooled':p,'zero_loss':zero,'boundary':boundary},indent=2,allow_nan=False)+chr(10))
print(p);print('Нулевые потери НЕ удостоверяют допустимость.')

Тот же событийный код, что у демонстрации: 3 поступления, 120 мс Fog, B=1. При 300 мс: задержки 120/239/потеря; при 200 мс один исход ещё не известен. Дополнительно проверяем адаптер на десяти коротких пилотных seed, не на утверждённых исследовательских окнах.

In [ ]:
c=Config(n_edge=1,n_fog=1,n_cloud=1,buffer=1,n_tasks=3,mobile_fraction=0,edge_fixed_ms=(0,0),fog_service_ms=(120,120),fog_cloud_ms=(0,0),cloud_service_ms=(0,0))
r=simulate(c,['.001','.002','.003'])
full=extract_cohort(0,r,(0,4_000_000,300_000_000));pending=extract_cohort(0,r,(0,4_000_000,200_000_000))
p=pooled_metrics([full]);assert abs(p['p95_ms']-233.05)<1e-10 and p['drop_fraction']==1/3 and p['H200']==1/3
assert pending.unfinished==1 and pooled_metrics([pending])['p95_ms'] is None
from dataclasses import asdict,replace
sampler=KernelSampler(replace(c,arrival_rate_s=1,buffer=2))
obs=[sampler(seed,.1,.2,.5) for seed in PILOT_SEEDS]
assert [o.full.seed for o in obs]==list(PILOT_SEEDS)
(out/'kernel-adapter.json').write_text(json.dumps({'scope':'short adapter regression, not pilot calibration','manual_complete':p,'manual_pending':asdict(pending),'observations':[asdict(o) for o in obs]},indent=2)+chr(10))
print('233,05 мс; потери 1/3; H200 1/3. Цензурированная когорта не даёт итогового p95.')

Контроллер получает заранее заданные синтетические наблюдения, а не поддельные результаты симуляции. Проверяем выбор более длинного прогрева, остановку на нулевых потерях и отделение итоговых seed. Исследовательские окна здесь не проигрываются.

In [ ]:
from test_pilot import FakeSampler
import tempfile
stable=select_windows(FakeSampler())
zero=select_windows(FakeSampler('zero'))
assert stable['status']=='READY' and stable['plan']['warmup']==4000
assert zero['status']=='INSUFFICIENT_DATA'
with tempfile.TemporaryDirectory() as tmp:
    final=run_frozen(FakeSampler(),stable,'main',journal=Path(tmp)/'main.json')
    confirmation=run_frozen(FakeSampler(),stable,'confirmation',journal=Path(tmp)/'confirm.json')
assert final['summary']['seed_ids']==list(MAIN_SEEDS)
assert confirmation['summary']['seed_ids']==list(CONFIRM_SEEDS)
(out/'controller-oracles.json').write_text(json.dumps({'scope':'supplied synthetic observations, not a simulated map','stable':stable,'zero':zero,'main':final,'confirmation':confirmation},indent=2,allow_nan=False)+chr(10))
print('Окна заморожены до итоговых seed; неуспех не запускает подбор следующего кандидата.')